# Time Series Basics

The most basic kind of time series object in pandas is a `Series` indexed by timestamps — which, outside of pandas, is often represented as Python strings or `datetime` objects (the building blocks covered in the previous notebook).

In [1]:
from datetime import datetime
import pandas as pd 
import numpy as np 

dates = [datetime(2011, 1, 2), datetime(2011, 1, 5),
        datetime(2011, 1, 7), datetime(2011, 1, 8),
        datetime(2011, 1, 10), datetime(2011, 1, 12)]

ts = pd.Series(np.random.standard_normal(6), index=dates)

ts

2011-01-02   -0.460027
2011-01-05    0.621039
2011-01-07    1.146416
2011-01-08   -0.482319
2011-01-10   -0.006083
2011-01-12   -0.365521
dtype: float64

Notice what pandas did with the index automatically: a list of `datetime` objects becomes a **`DatetimeIndex`** — pandas' specialized array type for timestamps, teased at the end of the previous notebook.

In [2]:
ts.index

DatetimeIndex(['2011-01-02', '2011-01-05', '2011-01-07', '2011-01-08',
               '2011-01-10', '2011-01-12'],
              dtype='datetime64[us]', freq=None)

Like other Series, arithmetic operations between differently indexed time series automatically align on the dates:

In [3]:
ts + ts[::2]

2011-01-02   -0.920054
2011-01-05         NaN
2011-01-07    2.292832
2011-01-08         NaN
2011-01-10   -0.012167
2011-01-12         NaN
dtype: float64

(Recall that `ts[::2]` selects every second element in `ts`; the misaligned dates on the other side of `+` produce `NaN`, the same as any other Series arithmetic.)

pandas stores timestamps using NumPy's `datetime64` dtype. **Nuance:** older material (and older pandas) will tell you this is always nanosecond resolution (`datetime64[ns]`). As of pandas 2.0+, resolution is flexible — pandas picks a resolution based on the precision of your input, from seconds up to nanoseconds. Since `datetime.datetime` objects only carry microsecond precision, that's what you get here:

In [4]:
ts.index.dtype

dtype('<M8[us]')

If you specifically need nanosecond resolution (e.g. for high-frequency financial tick data), you can force it with `.as_unit("ns")`:

```python
ts.index.as_unit("ns").dtype   # dtype('<M8[ns]')
```

Scalar values from a `DatetimeIndex` are pandas `Timestamp` objects:

In [5]:
stamp = ts.index[0]

stamp

Timestamp('2011-01-02 00:00:00')

A pandas `Timestamp` can be substituted almost anywhere you'd use a `datetime` object. The reverse is not true, however: `pandas.Timestamp` can store precision as fine as nanoseconds (when the source data supports it), while `datetime` only ever stores up to microseconds. `pandas.Timestamp` can also carry frequency information (if any) and knows how to do time zone conversions and other manipulations — more on both of these in the notebooks that follow.

---
## Indexing, Selection, Subsetting

A time series behaves like any other Series when you're indexing and selecting data based on the label:

In [6]:
stamp = ts.index[2]

ts[stamp]

np.float64(1.1464158030844789)

As a convenience, you can also pass a string that's interpretable as a date — pandas parses it the same way `pandas.to_datetime` would (see the previous notebook), so most reasonable formats work:

In [7]:
ts["2011-01-10"]

np.float64(-0.006083250565619808)

## Partial-String Indexing

For longer time series, you don't need the full timestamp — a year, or a year and month, is enough to select a whole slice of data at once. This is called **partial-string indexing**. `longer_ts` below is built with `pandas.date_range` (a tool for generating evenly spaced date sequences, covered in depth in the next notebook — here it's just a convenient way to build a long example):

In [8]:
longer_ts = pd.Series(np.random.standard_normal(1000), 
                    index = pd.date_range("2001-01-01", periods=1000))

longer_ts

2001-01-01    0.904966
2001-01-02    0.094669
2001-01-03    0.548698
2001-01-04    1.533440
2001-01-05   -0.730154
                ...   
2003-09-23   -0.249015
2003-09-24   -1.050213
2003-09-25   -0.295958
2003-09-26   -0.517217
2003-09-27    0.295976
Freq: D, Length: 1000, dtype: float64

Here the string "2001" is interpreted as a year and selects the time period. This also works if you specify the month:

In [9]:
longer_ts["2001-05"]

2001-05-01   -1.206452
2001-05-02   -0.097093
2001-05-03    1.365444
2001-05-04   -0.370115
2001-05-05   -0.347630
2001-05-06   -0.147019
2001-05-07   -0.438483
2001-05-08   -1.114955
2001-05-09    1.295867
2001-05-10   -1.281049
2001-05-11    0.120733
2001-05-12    0.336474
2001-05-13   -0.825448
2001-05-14    2.176643
2001-05-15   -1.224451
2001-05-16   -0.001272
2001-05-17    0.283022
2001-05-18    1.618139
2001-05-19   -0.104281
2001-05-20   -0.351209
2001-05-21    2.050123
2001-05-22    0.241386
2001-05-23   -0.114955
2001-05-24    0.777894
2001-05-25    0.091104
2001-05-26   -1.140914
2001-05-27   -0.018204
2001-05-28    1.462050
2001-05-29    0.427544
2001-05-30    1.422782
2001-05-31   -0.113950
Freq: D, dtype: float64

Slicing with `datetime` objects works as well:

In [10]:
ts[datetime(2011, 1, 7):]

2011-01-07    1.146416
2011-01-08   -0.482319
2011-01-10   -0.006083
2011-01-12   -0.365521
dtype: float64

In [11]:
ts[datetime(2011, 1, 7):datetime(2011, 1, 10)]

2011-01-07    1.146416
2011-01-08   -0.482319
2011-01-10   -0.006083
dtype: float64

**Gotcha:** range slicing like this relies on the index being sorted. If your `DatetimeIndex` isn't monotonic (increasing or decreasing), pandas will raise a `KeyError` rather than guess what you meant — sort it first with `ts.sort_index()` if needed.

As before, you can pass a string date, `datetime`, or `Timestamp`. Remember that slicing in this manner produces **views** on the source time series, like slicing NumPy arrays — no data is copied, and modifications to the slice will be reflected in the original. If you need an independent copy, call `.copy()` explicitly on the slice.

There is an equivalent instance method, `truncate`, that slices a Series between two dates:

In [12]:
ts.truncate(after="2011-01-09")

2011-01-02   -0.460027
2011-01-05    0.621039
2011-01-07    1.146416
2011-01-08   -0.482319
dtype: float64

All of this holds true for a DataFrame as well, indexing on its rows. **Gotcha:** you need `.loc` here — `long_df["2001-05"]` would raise a `KeyError`, because bracket indexing on a DataFrame looks for a *column* named `"2001-05"` by default. `.loc` is what tells pandas you mean row-based (label) indexing:

In [13]:
dates = pd.date_range("2000-01-01", periods=100, freq="W-WED")

long_df = pd.DataFrame(np.random.standard_normal((100, 4)),
                    index=dates, 
                    columns=["Colorado", "Texas", "New York", "Ohio"])

long_df.loc["2001-05"]

,Colorado,Texas,New York,Ohio
2001-05-02,-1.587494,-0.105460,-0.357500,-0.505278
2001-05-09,-0.383963,2.962971,-1.063027,1.202945
2001-05-16,1.356156,-0.619665,1.701634,0.647390
2001-05-23,-0.008686,0.083439,0.103903,-0.952620
2001-05-30,0.330419,0.322795,1.067787,-0.281753


## Time Series with Duplicate Indices

In some applications, there may be multiple data observations falling on the same timestamp — e.g. several sensor readings logged in the same second, or overlapping data pulled in from more than one source. Here is an example:

In [14]:
dates = pd.DatetimeIndex(["2000-01-01", "2000-01-02", "2000-01-02", "2000-01-02", "2000-01-03"])

dup_ts = pd.Series(np.arange(5), index=dates)

dup_ts

2000-01-01    0
2000-01-02    1
2000-01-02    2
2000-01-02    3
2000-01-03    4
dtype: int64

We can tell that the index is not unique by checking its `is_unique` property (to see exactly *which* labels are duplicated, `dates.duplicated()` returns a boolean mask):

In [15]:
dup_ts.index.is_unique


False

Indexing into this time series will now either produce scalar values or slices, depending on whether a timestamp is duplicated:

In [16]:
dup_ts["2000-01-03"] # not duplicated



np.int64(4)

In [17]:
dup_ts["2000-01-02"] # duplicated

2000-01-02    1
2000-01-02    2
2000-01-02    3
dtype: int64

Suppose you wanted to aggregate the data having nonunique timestamps. One way to do this is to use `groupby` and pass `level=0` (the one and only level) — the same "group by index level" trick from the Data Aggregation and Group Operations chapter, just applied to a `DatetimeIndex` instead of a `MultiIndex`:

In [18]:
grouped = dup_ts.groupby(level=0)

grouped.mean()

2000-01-01    0.0
2000-01-02    2.0
2000-01-03    4.0
dtype: float64

In [19]:
grouped.count()

2000-01-01    1
2000-01-02    3
2000-01-03    1
dtype: int64

---

## Summary / Cheat Sheet

**Ways to select from a time-indexed Series/DataFrame:**

| What you want | How |
|---|---|
| Exact timestamp | `ts[stamp]` or `ts["2011-01-10"]` |
| Whole year / month | `ts["2011"]` / `ts["2011-01"]` (partial-string indexing) |
| Range between two points | `ts[datetime(2011,1,7):]`, `ts.truncate(after=...)` |
| Same, but on a DataFrame's rows | `df.loc["2011-01"]` — plain `df["2011-01"]` looks for a *column*, not rows |
| Aggregate duplicate timestamps | `ts.groupby(level=0).mean()` |

**Nuances worth remembering:**
- A `DatetimeIndex`'s resolution isn't always nanoseconds anymore (pandas 2.0+) — it adapts to your input's precision; use `.as_unit("ns")` if you need to force it.
- Range slicing (`ts[a:b]`) needs a sorted (monotonic) index, or pandas raises `KeyError` instead of guessing.
- Label-based slices are **views**, not copies — mutate one and you mutate the original, unless you `.copy()` first.
- A duplicate `DatetimeIndex` (`is_unique == False`) makes indexing return either a scalar or a Series depending on whether that particular label repeats — worth checking `is_unique` before assuming one or the other.

**Up next:** *Date Ranges, Frequencies, and Shifting* — generating regular date sequences with `date_range` (used here just to build examples), assigning a `freq` to a time series, and shifting data forward/backward in time.